# Week 10 Live Coding: Is the Turnout Model Any Good?

A vendor sold us a turnout model and a probability for every voter. Today we (1) **build** a model like it — which is just a regression — (2) read its coefficients, and (3) check whether its probabilities can be **trusted**, using a *calibration plot* and a *Brier score*.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy — work in that tab. Edits you make to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

df = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/main/weeks/wk10_forecasts_and_calibration/data/turnout_history.csv')
df.head()

Each row is a registered voter. `voted08`...`voted14` are 0/1 (did they vote that year); plus `age`, `female`, and race indicators. We'll predict **`voted14`** (turnout in the 2014 midterm) from their earlier history. (This is real data from a 2014 field study — the years are historical, but building the model is exactly what a present-day vendor does.)

## Part 1: Build the turnout model (it's a regression)

First, a **train/test split**. We fit the model on 70% of voters (the *training* set) and keep the other 30% (the *test* set) hidden, so we can later check the model on voters it never saw. Grading a model on its own training data proves nothing.

In [ ]:
train = df.sample(frac=0.7, random_state=10)
test  = df.drop(train.index)
print('train:', len(train), 'voters, turnout', round(train['voted14'].mean(), 3))
print('test: ', len(test),  'voters, turnout', round(test['voted14'].mean(), 3))

In [ ]:
# The turnout model: same smf.ols you've used since Week 2 -- only the purpose is new.
model = smf.ols('voted14 ~ voted12 + voted10 + voted08 + age + female', data=train).fit()
print(model.params.round(4))
print('R-squared:', round(model.rsquared, 3))

**Read the coefficients like any regression table.** The coefficient on `voted12` is about **+0.34**: voting in 2012 is associated with a **34-percentage-point** higher chance of voting in 2014, holding the others fixed. `voted10` adds another ~18 points. `age` adds ~0.2 points per year; `female` is essentially zero (no gender gap here). **Past voting is the engine of every commercial turnout model** — and this is the same kind of coefficient table you'll read in a published paper.

## Part 2: Predict each voter's probability

The fitted value is the predicted probability of voting — the vendor's "0.83."  

In [ ]:
test = test.copy()
test['pred'] = model.predict(test)

print('predicted probabilities range from', round(test['pred'].min(), 3),
      'to', round(test['pred'].max(), 3))
# one voter, like the case's voter X:
print('\nExample voter:')
print(test[['voted12', 'voted10', 'age', 'pred']].iloc[0])

Two things to notice. (1) The model never scores anyone above ~0.71 here — with this history, no one is a near-certain voter. (2) This is a **linear probability model**, so it *can* output probabilities below 0 or above 1 (impossible values). It's a useful wart — though note that on *this* data the predictions all land between about 0.03 and 0.71, so it never actually bites here (which is why the `.clip(0, 1)` we use later changes nothing — it's just insurance). Give the model a different dataset with stronger predictors and a score could dip below 0 or above 1. The point stands: the model won't stop itself from handing you nonsense, so you check.

## Part 3: Is it calibrated?

A probability of 0.6 should mean: *of all the voters scored around 0.6, about 60% actually vote.* Let's check — first one band by hand, then all of them.

In [ ]:
# Of the held-out voters scored between 0.55 and 0.65, what fraction actually voted?
band = (test['pred'] >= 0.55) & (test['pred'] < 0.65)
print('voters in the 0.55-0.65 band:', band.sum())
print('model says ~0.60; actual turnout in this band:', round(test.loc[band, 'voted14'].mean(), 3))

Two new tools, both one line each:
- **`pd.cut(...)`** sorts each prediction into a *bin*. `np.linspace(0, 1, 11)` makes 11 evenly spaced edges from 0 to 1 — and 11 edges make **10 bins** (0–0.1, 0.1–0.2, …, 0.9–1.0).
- **`groupby('bin').agg(...)`** then averages *within* each bin: the mean predicted probability, and the actual fraction who voted.

If the model is calibrated, those two columns should match, bin by bin.

In [ ]:
# Now all ten bands at once: bin the predictions, then compare mean predicted to actual.
# clip(0,1) is insurance against impossible probabilities; pd.cut sorts into 10 bins (11 edges)
test['bin'] = pd.cut(test['pred'].clip(0, 1), bins=np.linspace(0, 1, 11))
calibration = (test.groupby('bin', observed=True)
               .agg(mean_predicted=('pred', 'mean'),
                    actual_voted=('voted14', 'mean'),
                    n=('voted14', 'size'))
               .dropna())
print(calibration.round(3))

In [ ]:
# The calibration plot: mean predicted (x) vs actual fraction who voted (y).
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot([0, 1], [0, 1], '--', color='gray', label='perfect calibration (45 deg)')
ax.scatter(calibration['mean_predicted'], calibration['actual_voted'],
           s=calibration['n'] / 12, color='#0F4D92', zorder=3, label='model (dot size = # voters)')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Actual fraction who voted')
ax.set_title('Calibration of the turnout model (held-out voters)')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

In [ ]:
# Brier score: average squared error of the probabilities. Lower is better.
brier_model    = np.mean((test['pred'].clip(0, 1) - test['voted14'])**2)
brier_baseline = np.mean((test['voted14'].mean()    - test['voted14'])**2)  # always guess the base rate
print('Brier score (our model):       ', round(brier_model, 4))
print('Brier score (guess base rate): ', round(brier_baseline, 4))

**Verdict.** The dots hug the 45-degree line, and the Brier score (~0.14) beats the always-guess-the-base-rate baseline (~0.18). On voters it never saw, *the model's probabilities mean what they say.* When it says 0.6, about 60% turn out.

That is the evidence you'd want before targeting a field program on these scores — and exactly what the vendor's one-pager never showed you. In the problem set you'll see what happens when the same model meets a **different electorate**.